In [1]:
print(df_full.columns.tolist())

NameError: name 'df_full' is not defined

In [ ]:
pip install xgboost

In [2]:
import time
import numpy as np
import pandas as pd
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    raise ImportError(
        "StratifiedGroupKFold needs scikit-learn >= 1.0. "
        "Run: pip install -U scikit-learn in the eurostar env."
    )

In [3]:
MODEL_DF_FULL_PATH = (r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\model_df_full.parquet")
COMBINED_RAW_PATH = (r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\combined_df_finalcleaned.parquet")

TARGET = "question_recommendation_nps_a"

In [4]:
BASE_PATH = r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data"  # matches model_df_full.parquet's folder
out_path = rf"{BASE_PATH}\gbm_experiment_results_stage1.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

NameError: name 'results_df' is not defined

In [5]:
df_raw = pd.read_parquet(COMBINED_RAW_PATH)

service_cols = [
    col for col in df_raw.columns
    if "service" in col.lower() or "train" in col.lower()
]

date_cols = [
    col for col in df_raw.columns
    if "date" in col.lower()
]

print("Service candidates:", service_cols)
print("Date candidates:", date_cols)

Service candidates: ['metadata_class_of_service', 'metadata_class_of_service_code', 'metadata_train_number', 'metadata_train_type', 'metadata_train_type_code', 'question_on_board_the_train_v', 'question_assistance_service_v', 'question_overall_satisfaction_wifi_onboard_the_train', 'question_overall_satisfaction_comfort_onboard_the_train', 'question_overall_satisfaction_cleanliness_onboard_the_train', 'question_overall_satisfaction_overall_service_from_eurostar_staff', 'question_onboard_catering_met_quality_of_food_in_meal_service', 'question_onboard_catering_met_quality_of_beverages_in_meal_service', 'question_onboard_catering_met_staff_hospitality_during_meal_service', 'question_onboard_catering_met_range_of_food_available_in_meal_service', 'question_onboard_catering_met_range_of_drinks_available_in_meal_service', 'question_disruption_performan_information_on_board_the_train_e_g_from_staff_announcements', 'question_assistance_service', 'question_assistance_service_r', 'question_at_the

In [8]:

SERVICE_COL = "metadata_train_number"
DATE_COL = "departure_date"


In [9]:


N_FOLDS = 4
RANDOM_STATE = 42

MAX_N_ESTIMATORS = 1000
EARLY_STOPPING_ROUNDS = 50
INNER_VAL_FRACTION = 0.15  

FIXED_PARAMS = {"tree_method": "hist", "n_jobs": -1, "random_state": RANDOM_STATE}


COARSE_GRID = [
    {"max_depth": 2, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10},
    {"max_depth": 4, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10},
    {"max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 20},
    {"max_depth": 5, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 20},
]

In [10]:
df_full = pd.read_parquet(MODEL_DF_FULL_PATH)
df_raw = pd.read_parquet(COMBINED_RAW_PATH)

assert df_full.shape[0] == df_raw.shape[0], "Row count mismatch — do NOT join positionally"
df_full = df_full.reset_index(drop=True)
df_raw = df_raw.reset_index(drop=True)

if TARGET in df_raw.columns:
    align_check = (df_full[TARGET].values == df_raw[TARGET].values).mean()
    print(f"Row alignment check (target match): {align_check:.4f}  (must be 1.0 — stop if not)")

if "metadata_price" not in df_full.columns:
    df_full["metadata_price"] = df_raw["metadata_price"].values

for col in [SERVICE_COL, DATE_COL]:
    if col not in df_raw.columns:
        raise KeyError(f"'{col}' not found in combined_df_finalcleaned — fix SERVICE_COL/DATE_COL above")


Row alignment check (target match): 1.0000  (must be 1.0 — stop if not)


In [11]:
groups = (df_raw[SERVICE_COL].astype(str) + "_" + df_raw[DATE_COL].astype(str)).values
group_counts = pd.Series(groups).value_counts()
print(f"Unique journey groups: {group_counts.shape[0]} across {len(groups)} rows")
print(f"Rows per group — mean: {group_counts.mean():.2f}, median: {group_counts.median():.0f}, max: {group_counts.max()}")


Unique journey groups: 46322 across 178384 rows
Rows per group — mean: 3.85, median: 3, max: 40


In [12]:

ALWAYS_DROP = [
    TARGET,
    "arrival_delay_clipped", "total_compensation", "early_minutes",
    "cause_No_Delay", "cause_select_No_Delay",
    "delay_cause_group", "delay_cause_select",
    "disrup_type_encoded", "equipment_type",
    "pax_quartile_check",
]
ALWAYS_DROP_FOUND = [c for c in ALWAYS_DROP if c in df_full.columns]
ALWAYS_DROP_MISSING = [c for c in ALWAYS_DROP if c not in df_full.columns]

In [13]:
print("\n" )
print("DATA INTEGRITY CHECKS")
print(f"df_full shape: {df_full.shape}")

dup_cols = df_full.columns[df_full.columns.duplicated()].tolist()
print(f"\nDuplicate column names: {dup_cols if dup_cols else 'None'}")
if dup_cols:
    print("STOP.")



DATA INTEGRITY CHECKS
df_full shape: (178384, 168)

Duplicate column names: None


In [10]:
non_numeric = df_full.select_dtypes(exclude="number").columns.tolist()
non_numeric_handled = [c for c in non_numeric if c in ALWAYS_DROP_FOUND]

print(f"\nNon-numeric columns already covered by ALWAYS_DROP: {non_numeric_handled or 'None'}")


Non-numeric columns already covered by ALWAYS_DROP (fine, gets removed below): ['delay_cause_group', 'delay_cause_select']


In [14]:
nulls = df_full.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(f"\nColumns with nulls ({len(nulls)} total):")
print(nulls.to_string() if len(nulls) else "None")



Columns with nulls (0 total):
None


In [15]:
pm_must_have = ["pm_days_since_toilet", "pm_days_since_climate", "pm_days_since_comms", "pm_days_since_interior", "pm_days_since_catering"]
print("\npm_days_since_* family — confirming no pre-drop happened this round:")
for c in pm_must_have:
    print(f"  {c}: {'present' if c in df_full.columns else 'MISSING — check upstream drop_cols'}")



pm_days_since_* family — confirming no pre-drop happened this round:
  pm_days_since_toilet: present
  pm_days_since_climate: present
  pm_days_since_comms: present
  pm_days_since_interior: present
  pm_days_since_catering: present


In [16]:
if "arrival_delay_minute" in df_full.columns:
    before = df_full["arrival_delay_minute"].copy()
    df_full["arrival_delay_minute"] = df_full["arrival_delay_minute"].clip(lower=0, upper=343)
    n_clipped = (before != df_full["arrival_delay_minute"]).sum()
    print(f"\narrival_delay_minute clipped to [0, 343]: {n_clipped} rows affected")


arrival_delay_minute clipped to [0, 343]: 34432 rows affected


## Targets

In [17]:
def nps_category(score):
    if score <= 6:
        return 0  # Detractor
    elif score <= 8:
        return 1  # Passive
    return 2  # Promoter

y_reg = df_full[TARGET].values.astype(float)
y_class = df_full[TARGET].apply(nps_category).values

class_dist = pd.Series(y_class).value_counts(normalize=True).sort_index() * 100
print(f"\nNPS class distribution — Detractor: {class_dist.get(0, 0):.2f}%, "
      f"Passive: {class_dist.get(1, 0):.2f}%, Promoter: {class_dist.get(2, 0):.2f}%")



NPS class distribution — Detractor: 16.61%, Passive: 32.42%, Promoter: 50.98%


## Feature-set staircase

In [19]:


print(f"\nALWAYS_DROP — found and removed: {ALWAYS_DROP_FOUND}")
print(f"ALWAYS_DROP — not present (fine, just noting): {ALWAYS_DROP_MISSING}")

all_cols = [c for c in df_full.columns if c not in ALWAYS_DROP_FOUND]

SATISFACTION_SUBITEMS = [
    "question_overall_satisfaction_booking_experience",
    "question_overall_satisfaction_cleanliness_onboard_the_train",
    "question_overall_satisfaction_comfort_onboard_the_train",
    "question_overall_satisfaction_experience_at_departure_station",
    "question_overall_satisfaction_information_provided_to_you_before_travelling",
    "question_overall_satisfaction_journey_punctuality",
    "question_overall_satisfaction_overall_service_from_eurostar_staff",
    "question_overall_satisfaction_wifi_onboard_the_train",
    "question_staff_manners_staff_were_friendly_and_welcoming",
    "question_staff_manners_staff_were_helpful",
    "question_staff_manners_staff_were_visible",
    "question_strategic_pillar_ass_travelling_by_eurostar_is_an_environmentally_sustainable_option",
    "question_ticket_price_satisfa",
]  

PROFILE_COLS = [
    "pct_adult", "pct_youth", "pct_child", "pct_senior",
    "avg_group_size", "group_pax_ratio", "total_pax",
    "is_frequent_traveller", "age_encoded", "residence_encoded", "language_encoded",
    "is_business", "is_leisure", "is_commuter", "is_vfr", "trips_l12m",
    "reason_quickest", "reason_prefers_train", "reason_eco",
    "reason_no_transfers", "reason_cheapest", "reason_quality",
    "reason_no_choice", "reason_loyalty_points", "reason_other",
]

TEMPORAL_TRIP_COLS = [
    "class_encoded",
    "departure_month", "departure_weekday", "is_weekend",
    "departure_hour_int", "departure_season_encoded",
    "time_of_day_encoded", "is_peak_hour", "has_connection",
]

PRICE_COL = ["metadata_price"]
STAFF_COL = ["staff_composite"]

BLOCKS = {
    "satisfaction": SATISFACTION_SUBITEMS, "profile": PROFILE_COLS,
    "temporal_trip": TEMPORAL_TRIP_COLS, "price": PRICE_COL, "staff": STAFF_COL,
}
for name, cols in BLOCKS.items():
    missing = [c for c in cols if c not in all_cols]
    if missing:
        print(f"\n[{name}] block — columns NOT found in df_full (check naming): {missing}")
    BLOCKS[name] = [c for c in cols if c in all_cols]

operational_only = [c for c in all_cols if c not in
                     BLOCKS["satisfaction"] + BLOCKS["profile"] + BLOCKS["temporal_trip"]
                     + BLOCKS["price"] + BLOCKS["staff"]]
just_price = operational_only + BLOCKS["price"]
price_and_staff = just_price + BLOCKS["staff"]
satisfaction_excluded = [c for c in all_cols if c not in BLOCKS["satisfaction"]]
profile_excluded = [c for c in satisfaction_excluded if c not in BLOCKS["profile"]]
everything = all_cols

FEATURE_SETS = {
    "operational_only": sorted(set(operational_only)),
    "just_price": sorted(set(just_price)),
    "price_and_staff": sorted(set(price_and_staff)),
    "profile_excluded": sorted(set(profile_excluded)),
    "satisfaction_excluded": sorted(set(satisfaction_excluded)),
    "everything": sorted(set(everything)),
}

print("\nFeature set sizes:")
for name, cols in FEATURE_SETS.items():
    print(f"  {name}: {len(cols)} features")
print("\noperational_only columns (visual QA — flag anything that looks misplaced):")
print(FEATURE_SETS["operational_only"])


ALWAYS_DROP — found and removed: ['question_recommendation_nps_a', 'arrival_delay_clipped', 'early_minutes', 'cause_No_Delay', 'cause_select_No_Delay', 'delay_cause_group', 'delay_cause_select', 'disrup_type_encoded', 'pax_quartile_check']
ALWAYS_DROP — not present (fine, just noting): ['total_compensation', 'equipment_type']

[profile] block — columns NOT found in df_full (check naming): ['residence_encoded', 'language_encoded']

Feature set sizes:
  operational_only: 112 features
  just_price: 113 features
  price_and_staff: 114 features
  profile_excluded: 123 features
  satisfaction_excluded: 146 features
  everything: 159 features

operational_only columns (visual QA — flag anything that looks misplaced):
['arrival_delay_minute', 'arrived_early', 'average_days_since_last_exams', 'average_days_since_last_exams_has_data', 'average_days_since_last_exams_resid', 'average_days_since_last_exams_was_tracked', 'cause_Eurostar_Operations', 'cause_Infrastructure', 'cause_Passenger_Related'

In [20]:
def fit_with_early_stopping(model_cls, X_train, y_train, groups_train, params, fit_kwargs):

    gss = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_FRACTION, random_state=RANDOM_STATE)
    inner_tr, inner_val = next(gss.split(X_train, y_train, groups_train))

    probe = model_cls(n_estimators=MAX_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                       **params, **fit_kwargs.get("init_kwargs", {}))
    probe.fit(X_train.iloc[inner_tr], y_train[inner_tr],
              eval_set=[(X_train.iloc[inner_val], y_train[inner_val])], verbose=False)
    best_n = probe.best_iteration + 1

    final_model = model_cls(n_estimators=best_n, **params, **fit_kwargs.get("init_kwargs", {}))
    final_model.fit(X_train, y_train, verbose=False)
    return final_model, best_n

In [21]:
def run_cv_regression(X, y, groups, params, n_folds=N_FOLDS):
    gkf = GroupKFold(n_splits=n_folds)
    test_r2s, test_rmses, train_r2s, mins, best_ns = [], [], [], [], []
    for train_idx, test_idx in gkf.split(X, y, groups):
        t0 = time.time()
        X_train, y_train, g_train = X.iloc[train_idx], y[train_idx], groups[train_idx]
        X_test, y_test = X.iloc[test_idx], y[test_idx]

        model, best_n = fit_with_early_stopping(
            XGBRegressor, X_train, y_train, g_train, params,
            fit_kwargs={"init_kwargs": {"eval_metric": "rmse"}},
        )
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        test_r2s.append(r2_score(y_test, test_pred))
        test_rmses.append(mean_squared_error(y_test, test_pred) ** 0.5)
        train_r2s.append(r2_score(y_train, train_pred))
        best_ns.append(best_n)
        mins.append((time.time() - t0) / 60)

    test_r2_mean = np.mean(test_r2s)
    return {
        "test_r2_mean": test_r2_mean, "test_r2_std": np.std(test_r2s),
        "test_rmse_mean": np.mean(test_rmses), "test_rmse_std": np.std(test_rmses),
        "train_r2_mean": np.mean(train_r2s),
        "gap_mean": np.mean(train_r2s) - test_r2_mean,
        "gap_std": np.std(np.array(train_r2s) - np.array(test_r2s)),
        "best_n_estimators_mean": np.mean(best_ns),
        "minutes_per_fold": np.mean(mins),
    }

def run_cv_classification(X, y, groups, params, n_folds=N_FOLDS):
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    test_accs, train_accs, mins, best_ns = [], [], [], []
    aucs = {0: [], 1: [], 2: []}  # Detractor, Passive, Promoter
    for train_idx, test_idx in sgkf.split(X, y, groups):
        t0 = time.time()
        X_train, y_train, g_train = X.iloc[train_idx], y[train_idx], groups[train_idx]
        X_test, y_test = X.iloc[test_idx], y[test_idx]

        model, best_n = fit_with_early_stopping(
            XGBClassifier, X_train, y_train, g_train, params,
            fit_kwargs={"init_kwargs": {"eval_metric": "mlogloss", "objective": "multi:softprob"}},
        )
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)

        test_accs.append(accuracy_score(y_test, test_pred))
        train_accs.append(accuracy_score(y_train, train_pred))
        best_ns.append(best_n)
        for cls in [0, 1, 2]:
            y_bin = (y_test == cls).astype(int)
            if 0 < y_bin.sum() < len(y_bin):
                aucs[cls].append(roc_auc_score(y_bin, proba[:, cls]))
            else:
                aucs[cls].append(np.nan)
        mins.append((time.time() - t0) / 60)

    test_acc_mean = np.mean(test_accs)
    return {
        "test_accuracy_mean": test_acc_mean, "test_accuracy_std": np.std(test_accs),
        "train_accuracy_mean": np.mean(train_accs),
        "gap_mean": np.mean(train_accs) - test_acc_mean,
        "gap_std": np.std(np.array(train_accs) - np.array(test_accs)),
        "auc_detractor_mean": np.nanmean(aucs[0]),
        "auc_passive_mean": np.nanmean(aucs[1]),
        "auc_promoter_mean": np.nanmean(aucs[2]),
        "best_n_estimators_mean": np.mean(best_ns),
        "minutes_per_fold": np.mean(mins),
    }

## Timing probe

In [22]:
print("\n")
print("TIMING PROBE (one config, 'everything', both targets, full CV)")
probe_params = {"max_depth": 4, "learning_rate": 0.05, "subsample": 0.8,
                 "colsample_bytree": 0.8, "min_child_weight": 10, **FIXED_PARAMS}
X_probe = df_full[FEATURE_SETS["everything"]]
n_combos = len(COARSE_GRID) * len(FEATURE_SETS)

t0 = time.time()
_ = run_cv_regression(X_probe, y_reg, groups, probe_params, n_folds=N_FOLDS)
probe_minutes = (time.time() - t0) / 60
print(f"Full regression CV (largest dataset, {N_FOLDS} folds, early stopping): {probe_minutes:.1f} min")

t0 = time.time()
_ = run_cv_classification(X_probe, y_class, groups, probe_params, n_folds=N_FOLDS)
clf_probe_minutes = (time.time() - t0) / 60
print(f"Full classification CV (largest dataset): {clf_probe_minutes:.1f} min")

est_total = (probe_minutes + clf_probe_minutes) * n_combos
print(f"Stage 1 = {n_combos} configs x 6 sets x 2 targets -> estimate: {est_total:.0f} min ({est_total/60:.1f} hrs)")




TIMING PROBE (one config, 'everything', both targets, full CV)
Full regression CV (largest dataset, 4 folds, early stopping): 4.5 min
Full classification CV (largest dataset): 9.8 min
Stage 1 = 24 configs x 6 sets x 2 targets -> estimate: 343 min (5.7 hrs)


In [26]:
results = []
for set_name, cols in FEATURE_SETS.items():
    X_set = df_full[cols]
    for hp in COARSE_GRID:
        params = {**hp, **FIXED_PARAMS}
        reg_res = run_cv_regression(X_set, y_reg, groups, params)
        clf_res = run_cv_classification(X_set, y_class, groups, params)
        row = {"feature_set": set_name, "n_features": len(cols), **hp, **reg_res,
               **{f"clf_{k}": v for k, v in clf_res.items()}}
        results.append(row)
        print(f"[{set_name:22s} | depth={hp['max_depth']} lr={hp['learning_rate']} mcw={hp['min_child_weight']}] "
              f"R2 test={reg_res['test_r2_mean']:.4f}±{reg_res['test_r2_std']:.4f} "
              f"gap={reg_res['gap_mean']:.4f}  |  "
              f"Acc test={clf_res['test_accuracy_mean']:.4f}±{clf_res['test_accuracy_std']:.4f} "
              f"gap={clf_res['gap_mean']:.4f}  "
              f"({reg_res['minutes_per_fold'] + clf_res['minutes_per_fold']:.1f} min/fold)")

results_df = pd.DataFrame(results)
out_path = rf"{BASE_PATH}\gbm_experiment_results_stage1.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

[operational_only       | depth=2 lr=0.05 mcw=10] R2 test=0.1186±0.0067 gap=0.0064  |  Acc test=0.5382±0.0015 gap=0.0019  (0.9 min/fold)
[operational_only       | depth=4 lr=0.05 mcw=10] R2 test=0.1197±0.0072 gap=0.0141  |  Acc test=0.5384±0.0014 gap=0.0051  (0.5 min/fold)
[operational_only       | depth=3 lr=0.03 mcw=20] R2 test=0.1195±0.0068 gap=0.0087  |  Acc test=0.5381±0.0016 gap=0.0037  (1.2 min/fold)
[operational_only       | depth=5 lr=0.03 mcw=20] R2 test=0.1199±0.0076 gap=0.0227  |  Acc test=0.5380±0.0010 gap=0.0080  (1.1 min/fold)
[just_price             | depth=2 lr=0.05 mcw=10] R2 test=0.1269±0.0071 gap=0.0062  |  Acc test=0.5390±0.0015 gap=0.0024  (2.0 min/fold)
[just_price             | depth=4 lr=0.05 mcw=10] R2 test=0.1275±0.0074 gap=0.0164  |  Acc test=0.5392±0.0014 gap=0.0058  (1.1 min/fold)
[just_price             | depth=3 lr=0.03 mcw=20] R2 test=0.1275±0.0072 gap=0.0105  |  Acc test=0.5392±0.0010 gap=0.0039  (2.5 min/fold)
[just_price             | depth=5 lr=0.03

NameError: name 'BASE_PATH' is not defined

In [28]:
results_df = pd.DataFrame(results)
out_path = rf"{BASE_PATH}\gbm_experiment_results_stage1.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")


Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\gbm_experiment_results_stage1.csv


In [31]:

MAX_N_ESTIMATORS = 2000 

probe_configs = [
    ("satisfaction_excluded", {"max_depth": 2, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10}),
    ("satisfaction_excluded", {"max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 20}),
    ("everything",            {"max_depth": 2, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10}),
    ("everything",            {"max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 20}),
]

for set_name, hp in probe_configs:
    X_set = df_full[FEATURE_SETS[set_name]]
    params = {**hp, **FIXED_PARAMS}
    reg_res = run_cv_regression(X_set, y_reg, groups, params)
    print(f"[{set_name:22s} | depth={hp['max_depth']}] "
          f"R2={reg_res['test_r2_mean']:.4f}  best_n_estimators={reg_res['best_n_estimators_mean']:.0f} "
          f"(ceiling was {MAX_N_ESTIMATORS})")

MAX_N_ESTIMATORS = 1000 

[satisfaction_excluded  | depth=2] R2=0.3259  best_n_estimators=1030 (ceiling was 2000)
[satisfaction_excluded  | depth=3] R2=0.3297  best_n_estimators=1061 (ceiling was 2000)
[everything             | depth=2] R2=0.5884  best_n_estimators=1845 (ceiling was 2000)
[everything             | depth=3] R2=0.5956  best_n_estimators=1967 (ceiling was 2000)


## summary three criteria side by side, sorted by test performance.

In [29]:
.
print("\n--- Regression: all configs, sorted by test R² (per feature set) ---")
reg_cols = ["feature_set", "max_depth", "learning_rate", "min_child_weight",
            "test_r2_mean", "test_r2_std", "train_r2_mean", "gap_mean", "best_n_estimators_mean"]
print(results_df.sort_values(["feature_set", "test_r2_mean"], ascending=[True, False])[reg_cols].to_string(index=False))

print("\n--- Classification: all configs, sorted by test accuracy (per feature set) ---")
clf_cols = ["feature_set", "max_depth", "learning_rate", "min_child_weight",
            "clf_test_accuracy_mean", "clf_test_accuracy_std", "clf_train_accuracy_mean", "clf_gap_mean",
            "clf_auc_detractor_mean", "clf_auc_passive_mean", "clf_auc_promoter_mean"]
print(results_df.sort_values(["feature_set", "clf_test_accuracy_mean"], ascending=[True, False])[clf_cols].to_string(index=False))


--- Regression: all configs, sorted by test R² (per feature set) ---
          feature_set  max_depth  learning_rate  min_child_weight  test_r2_mean  test_r2_std  train_r2_mean  gap_mean  best_n_estimators_mean
           everything          5           0.03                20      0.599332     0.004235       0.638566  0.039234                   852.4
           everything          4           0.05                10      0.597513     0.004575       0.628741  0.031228                   822.6
           everything          3           0.03                20      0.592247     0.004923       0.602513  0.010267                   998.8
           everything          2           0.05                10      0.584178     0.004780       0.589701  0.005523                   999.6
           just_price          5           0.03                20      0.128127     0.007433       0.151065  0.022938                   258.2
           just_price          3           0.03                20      0.12750

In [30]:
probe_params = {**hp, **FIXED_PARAMS}
probe_params["n_estimators"] = 2000  # up from 1000, just for this check

In [32]:
STAGE2_DEPTHS_BY_SET = {
    "operational_only": [4],
    "price_and_staff": [2, 5],
}

def fine_grid(depths):
    return [
        {"max_depth": d, "learning_rate": lr, "subsample": 0.8,
         "colsample_bytree": 0.8, "min_child_weight": mcw}
        for d in depths
        for lr in [0.03, 0.04, 0.05]
        for mcw in [10, 15, 20]
    ]

results_stage2 = []
for set_name, depths in STAGE2_DEPTHS_BY_SET.items():
    X_set = df_full[FEATURE_SETS[set_name]]
    for hp in fine_grid(depths):
        params = {**hp, **FIXED_PARAMS}
        reg_res = run_cv_regression(X_set, y_reg, groups, params)
        clf_res = run_cv_classification(X_set, y_class, groups, params)
        row = {"feature_set": set_name, "n_features": len(FEATURE_SETS[set_name]), **hp, **reg_res,
               **{f"clf_{k}": v for k, v in clf_res.items()}}
        results_stage2.append(row)
        print(f"[{set_name:16s} | depth={hp['max_depth']} lr={hp['learning_rate']} mcw={hp['min_child_weight']}] "
              f"R2 test={reg_res['test_r2_mean']:.4f}±{reg_res['test_r2_std']:.4f} "
              f"gap={reg_res['gap_mean']:.4f}  |  "
              f"Acc test={clf_res['test_accuracy_mean']:.4f}±{clf_res['test_accuracy_std']:.4f} "
              f"gap={clf_res['gap_mean']:.4f}  "
              f"({reg_res['minutes_per_fold'] + clf_res['minutes_per_fold']:.1f} min/fold)")

results_stage2_df = pd.DataFrame(results_stage2)
out_path_stage2 = rf"{BASE_PATH}\gbm_experiment_results_stage2.csv"
results_stage2_df.to_csv(out_path_stage2, index=False)
print(f"\nSaved: {out_path_stage2}")

[operational_only | depth=4 lr=0.03 mcw=10] R2 test=0.1198±0.0071 gap=0.0151  |  Acc test=0.5381±0.0010 gap=0.0059  (0.7 min/fold)
[operational_only | depth=4 lr=0.03 mcw=15] R2 test=0.1197±0.0070 gap=0.0147  |  Acc test=0.5382±0.0012 gap=0.0058  (0.8 min/fold)
[operational_only | depth=4 lr=0.03 mcw=20] R2 test=0.1197±0.0071 gap=0.0138  |  Acc test=0.5382±0.0012 gap=0.0055  (0.7 min/fold)
[operational_only | depth=4 lr=0.04 mcw=10] R2 test=0.1198±0.0070 gap=0.0144  |  Acc test=0.5380±0.0011 gap=0.0056  (0.6 min/fold)
[operational_only | depth=4 lr=0.04 mcw=15] R2 test=0.1196±0.0070 gap=0.0155  |  Acc test=0.5381±0.0008 gap=0.0058  (0.7 min/fold)
[operational_only | depth=4 lr=0.04 mcw=20] R2 test=0.1197±0.0070 gap=0.0158  |  Acc test=0.5383±0.0008 gap=0.0055  (0.7 min/fold)
[operational_only | depth=4 lr=0.05 mcw=10] R2 test=0.1197±0.0072 gap=0.0141  |  Acc test=0.5384±0.0014 gap=0.0051  (0.5 min/fold)
[operational_only | depth=4 lr=0.05 mcw=15] R2 test=0.1196±0.0069 gap=0.0144  |  Ac

In [33]:

print("\n--- Stage 2 Regression: sorted by test R² (per feature set) ---")
reg_cols = ["feature_set", "max_depth", "learning_rate", "min_child_weight",
            "test_r2_mean", "test_r2_std", "train_r2_mean", "gap_mean", "best_n_estimators_mean"]
print(results_stage2_df.sort_values(["feature_set", "test_r2_mean"], ascending=[True, False])[reg_cols].to_string(index=False))

print("\n--- Stage 2 Classification: sorted by test accuracy (per feature set) ---")
clf_cols = ["feature_set", "max_depth", "learning_rate", "min_child_weight",
            "clf_test_accuracy_mean", "clf_test_accuracy_std", "clf_train_accuracy_mean", "clf_gap_mean",
            "clf_auc_detractor_mean", "clf_auc_passive_mean", "clf_auc_promoter_mean"]
print(results_stage2_df.sort_values(["feature_set", "clf_test_accuracy_mean"], ascending=[True, False])[clf_cols].to_string(index=False))


--- Stage 2 Regression: sorted by test R² (per feature set) ---
     feature_set  max_depth  learning_rate  min_child_weight  test_r2_mean  test_r2_std  train_r2_mean  gap_mean  best_n_estimators_mean
operational_only          4           0.03                10      0.119809     0.007109       0.134924  0.015116                   313.0
operational_only          4           0.04                10      0.119788     0.007015       0.134228  0.014441                   220.0
operational_only          4           0.05                10      0.119721     0.007200       0.133795  0.014074                   169.6
operational_only          4           0.03                20      0.119685     0.007064       0.133484  0.013799                   281.8
operational_only          4           0.03                15      0.119676     0.006994       0.134408  0.014732                   304.4
operational_only          4           0.04                20      0.119668     0.007018       0.135508  0.015841 

In [34]:
FINAL_TEST_FRACTION = 0.15 

FINAL_CONFIGS = {
    "operational_only": {"max_depth": 4, "learning_rate": 0.05, "subsample": 0.8,
                          "colsample_bytree": 0.8, "min_child_weight": 10},
    "price_and_staff":  {"max_depth": 2, "learning_rate": 0.05, "subsample": 0.8,
                          "colsample_bytree": 0.8, "min_child_weight": 15},
}

final_models = {}   
final_holdouts = {}  
final_summary = []

for set_name, hp in FINAL_CONFIGS.items():
    X_set = df_full[FEATURE_SETS[set_name]]
    params = {**hp, **FIXED_PARAMS}

    gss = GroupShuffleSplit(n_splits=1, test_size=FINAL_TEST_FRACTION, random_state=RANDOM_STATE)
    train_idx, holdout_idx = next(gss.split(X_set, y_reg, groups))

    X_train, X_holdout = X_set.iloc[train_idx], X_set.iloc[holdout_idx]
    y_reg_train, y_reg_holdout = y_reg[train_idx], y_reg[holdout_idx]
    y_class_train, y_class_holdout = y_class[train_idx], y_class[holdout_idx]
    g_train = groups[train_idx]

    reg_model, reg_best_n = fit_with_early_stopping(
        XGBRegressor, X_train, y_reg_train, g_train, params,
        fit_kwargs={"init_kwargs": {"eval_metric": "rmse"}},
    )
    clf_model, clf_best_n = fit_with_early_stopping(
        XGBClassifier, X_train, y_class_train, g_train, params,
        fit_kwargs={"init_kwargs": {"eval_metric": "mlogloss", "objective": "multi:softprob"}},
    )

    holdout_r2 = r2_score(y_reg_holdout, reg_model.predict(X_holdout))
    holdout_acc = accuracy_score(y_class_holdout, clf_model.predict(X_holdout))

    final_models[(set_name, "reg")] = reg_model
    final_models[(set_name, "clf")] = clf_model
    final_holdouts[set_name] = {"X_holdout": X_holdout, "y_reg_holdout": y_reg_holdout,
                                 "y_class_holdout": y_class_holdout}
    final_summary.append({"feature_set": set_name, **hp,
                           "holdout_r2": holdout_r2, "holdout_accuracy": holdout_acc,
                           "reg_best_n_estimators": reg_best_n, "clf_best_n_estimators": clf_best_n,
                           "n_train": len(train_idx), "n_holdout": len(holdout_idx)})

    print(f"[{set_name}] holdout R2={holdout_r2:.4f}  holdout Acc={holdout_acc:.4f}  "
          f"(n_holdout={len(holdout_idx)}, reg_n_est={reg_best_n}, clf_n_est={clf_best_n})")

final_summary_df = pd.DataFrame(final_summary)
out_path_final = rf"{BASE_PATH}\gbm_final_holdout_results.csv"
final_summary_df.to_csv(out_path_final, index=False)
print(f"\nSaved: {out_path_final}")

[operational_only] holdout R2=0.1157  holdout Acc=0.5420  (n_holdout=27112, reg_n_est=235, clf_n_est=203)
[price_and_staff] holdout R2=0.2572  holdout Acc=0.5717  (n_holdout=27112, reg_n_est=554, clf_n_est=650)

Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\gbm_final_holdout_results.csv


In [36]:
for seed in [101, 202]:
    gss_check = GroupShuffleSplit(n_splits=1, test_size=FINAL_TEST_FRACTION, random_state=seed)
    X_set = df_full[FEATURE_SETS["price_and_staff"]]
    train_idx, holdout_idx = next(gss_check.split(X_set, y_reg, groups))
    X_train, X_holdout = X_set.iloc[train_idx], X_set.iloc[holdout_idx]
    y_reg_train, y_reg_holdout = y_reg[train_idx], y_reg[holdout_idx]
    g_train = groups[train_idx]
    params = {**FINAL_CONFIGS["price_and_staff"], **FIXED_PARAMS}
    reg_model_check, _ = fit_with_early_stopping(
        XGBRegressor, X_train, y_reg_train, g_train, params,
        fit_kwargs={"init_kwargs": {"eval_metric": "rmse"}},
    )
    r2_check = r2_score(y_reg_holdout, reg_model_check.predict(X_holdout))
    print(f"seed={seed}: holdout R2={r2_check:.4f}")

seed=101: holdout R2=0.2772
seed=202: holdout R2=0.2791


In [37]:
print(" Feature importances (gain-based, XGBoost default) — full list \n")

importance_rows = []
for set_name in FINAL_CONFIGS:
    for target in ["reg", "clf"]:
        model = final_models[(set_name, target)]
        importances = pd.Series(model.feature_importances_, index=FEATURE_SETS[set_name]).sort_values(ascending=False)
        print(f"--- {set_name} | {target} ({len(importances)} features) ---")
        print(importances.to_string())
        print()
        for feat, val in importances.items():
            importance_rows.append({"feature_set": set_name, "target": target, "feature": feat, "importance": val})

importance_df = pd.DataFrame(importance_rows)
out_path_importance = rf"{BASE_PATH}\feature_importances_full.csv"
importance_df.to_csv(out_path_importance, index=False)
print(f"Saved: {out_path_importance}")

 Feature importances (gain-based, XGBoost default) — full list 

--- operational_only | reg (112 features) ---
delay_gt30                                   0.373635
delay_gt15                                   0.178477
delay_gt60                                   0.075833
arrival_delay_minute                         0.040544
delay_gt10                                   0.038541
is_channel                                   0.026946
had_disruption                               0.025291
equip_TGH                                    0.023645
compensation_liability_evouchers             0.022292
compensation_liability_cash                  0.012946
cm_open_wifi_resid                           0.010027
pm_days_since_toilet_has_data                0.009727
early_journey_delay_minute                   0.009095
delay_is_internal                            0.004134
has_equipment_change                         0.004075
delay_is_external                            0.003444
cm_open_toilet           

## SHAP

In [23]:
import shap 
SHAP_SAMPLE_SIZE = 3000
SHAP_INTERACTION_SAMPLE_SIZE = 800

shap_pairs_all = []

for set_name in FINAL_CONFIGS:
    X_holdout = final_holdouts[set_name]["X_holdout"]
    reg_model = final_models[(set_name, "reg")]  

    explainer = shap.TreeExplainer(reg_model)

    sample_main = X_holdout.sample(n=min(SHAP_SAMPLE_SIZE, len(X_holdout)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample_main)
    mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=sample_main.columns).sort_values(ascending=False)
    print(f" {set_name}: top 10 features by mean |SHAP| ")
    print(mean_abs_shap.head(10).to_string())
    print()

    sample_int = X_holdout.sample(n=min(SHAP_INTERACTION_SAMPLE_SIZE, len(X_holdout)), random_state=RANDOM_STATE)
    interaction_values = explainer.shap_interaction_values(sample_int)

    cols = sample_int.columns
    pairs = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pairs.append({"feature_set": set_name, "feature_1": cols[i], "feature_2": cols[j],
                           "mean_abs_interaction": np.abs(interaction_values[:, i, j]).mean()})
    pairs_df = pd.DataFrame(pairs).sort_values("mean_abs_interaction", ascending=False)
    print(f" {set_name}: top 10 interaction pairs ")
    print(pairs_df.head(10).to_string(index=False))
    print()
    shap_pairs_all.append(pairs_df)

all_pairs_df = pd.concat(shap_pairs_all, ignore_index=True)
out_path_pairs = rf"{BASE_PATH}\shap_interaction_pairs_stage2.csv"
all_pairs_df.to_csv(out_path_pairs, index=False)
print(f"\nSaved: {out_path_pairs}")

NameError: name 'FINAL_CONFIGS' is not defined

In [39]:
CLASS_NAMES = {0: "Detractor", 1: "Passive", 2: "Promoter"}
clf_shap_rows = []

print(" Classifier SHAP: top 10 features by mean |SHAP|, per class \n")

for set_name in FINAL_CONFIGS:
    X_holdout = final_holdouts[set_name]["X_holdout"]
    clf_model = final_models[(set_name, "clf")]

    clf_explainer = shap.TreeExplainer(clf_model)
    sample_clf = X_holdout.sample(n=min(SHAP_SAMPLE_SIZE, len(X_holdout)), random_state=RANDOM_STATE)
    clf_shap_values = clf_explainer.shap_values(sample_clf)

    if isinstance(clf_shap_values, list):
        per_class = clf_shap_values
    else:
        arr = np.asarray(clf_shap_values)
        per_class = [arr[:, :, c] for c in range(len(CLASS_NAMES))]

    print(f" {set_name} ")
    for cls, vals in enumerate(per_class):
        mean_abs = pd.Series(np.abs(vals).mean(axis=0), index=sample_clf.columns).sort_values(ascending=False)
        for feat, val in mean_abs.items():
            clf_shap_rows.append({"feature_set": set_name, "class": CLASS_NAMES[cls],
                                   "feature": feat, "mean_abs_shap": val})
        print(f"  [{CLASS_NAMES[cls]}] top 10:")
        print(mean_abs.head(10).to_string())
        print()

clf_shap_df = pd.DataFrame(clf_shap_rows)
out_path_clf_shap = rf"{BASE_PATH}\shap_classifier_per_class.csv"
clf_shap_df.to_csv(out_path_clf_shap, index=False)
print(f"Saved: {out_path_clf_shap}")

=== Classifier SHAP: top 10 features by mean |SHAP|, per class ===

--- operational_only ---
  [Detractor] top 10:
arrival_delay_minute            0.167717
early_journey_delay_minute      0.091846
had_disruption                  0.058302
pm_days_since_catering          0.037725
delay_gt30                      0.035222
pm_days_since_comms             0.027168
pm_days_since_toilet_resid      0.023330
has_equipment_change            0.023281
pm_days_since_catering_resid    0.016483
delay_gt15                      0.015198

  [Passive] top 10:
equip_TGH                        0.020032
arrival_delay_minute             0.011646
early_journey_delay_minute       0.009972
pm_days_since_toilet             0.008020
clean_score_routine              0.006808
total_open_faults                0.006376
last_clean_score                 0.005901
arrived_early                    0.004459
average_days_since_last_exams    0.004287
cm_open_interior                 0.003561

  [Promoter] top 10:
arrival_dela

In [40]:
for seed in [101, 202]:
    gss_check = GroupShuffleSplit(n_splits=1, test_size=FINAL_TEST_FRACTION, random_state=seed)
    X_set = df_full[FEATURE_SETS["price_and_staff"]]
    train_idx, holdout_idx = next(gss_check.split(X_set, y_reg, groups))
    X_train, X_holdout = X_set.iloc[train_idx], X_set.iloc[holdout_idx]
    y_reg_train, y_reg_holdout = y_reg[train_idx], y_reg[holdout_idx]
    g_train = groups[train_idx]
    params = {**FINAL_CONFIGS["price_and_staff"], **FIXED_PARAMS}
    reg_model_check, _ = fit_with_early_stopping(
        XGBRegressor, X_train, y_reg_train, g_train, params,
        fit_kwargs={"init_kwargs": {"eval_metric": "rmse"}},
    )
    r2_check = r2_score(y_reg_holdout, reg_model_check.predict(X_holdout))
    print(f"seed={seed}: holdout R2={r2_check:.4f}")

seed=101: holdout R2=0.2772
seed=202: holdout R2=0.2791


In [41]:
CLF_INTERACTION_SAMPLE_SIZE = 500  

clf_shap_interaction_rows = []

for set_name in FINAL_CONFIGS:
    X_holdout = final_holdouts[set_name]["X_holdout"]
    clf_model = final_models[(set_name, "clf")]
    clf_explainer = shap.TreeExplainer(clf_model)

    sample_int = X_holdout.sample(n=min(CLF_INTERACTION_SAMPLE_SIZE, len(X_holdout)),
                                   random_state=RANDOM_STATE)
    interaction_values = clf_explainer.shap_interaction_values(sample_int)

   
    if isinstance(interaction_values, list):
        per_class_int = interaction_values
    else:
        arr = np.asarray(interaction_values)
        per_class_int = [arr[:, :, :, c] for c in range(len(CLASS_NAMES))]

    cols = sample_int.columns
    print(f" {set_name}: top interaction pairs by class \n")
    for cls, int_vals in enumerate(per_class_int):
        pairs = []
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                pairs.append({
                    "feature_set": set_name, "class": CLASS_NAMES[cls],
                    "feature_1": cols[i], "feature_2": cols[j],
                    "mean_abs_interaction": np.abs(int_vals[:, i, j]).mean(),
                })
        pairs_df = pd.DataFrame(pairs).sort_values("mean_abs_interaction", ascending=False)
        print(f" {CLASS_NAMES[cls]} ")
        print(pairs_df.head(10).to_string(index=False))
        print()
        clf_shap_interaction_rows.append(pairs_df)

clf_interaction_df = pd.concat(clf_shap_interaction_rows, ignore_index=True)
out_path_clf_int = rf"{BASE_PATH}\shap_interaction_pairs_classifier.csv"
clf_interaction_df.to_csv(out_path_clf_int, index=False)
print(f"Saved: {out_path_clf_int}")

 operational_only: top interaction pairs by class 

--- Detractor ---
     feature_set     class                  feature_1                  feature_2  mean_abs_interaction
operational_only Detractor       arrival_delay_minute             had_disruption              0.028918
operational_only Detractor       arrival_delay_minute early_journey_delay_minute              0.021148
operational_only Detractor early_journey_delay_minute             had_disruption              0.008903
operational_only Detractor                 delay_gt15             had_disruption              0.004890
operational_only Detractor                 delay_gt30 early_journey_delay_minute              0.004280
operational_only Detractor                  delay_gt5             had_disruption              0.003593
operational_only Detractor       arrival_delay_minute          delay_is_external              0.003426
operational_only Detractor       cm_open_toilet_resid             had_disruption              0.003310
ope

In [42]:
SEED_CHECK_SEEDS = list(range(101, 111)) 
seed_check_rows = []

for set_name, hp in FINAL_CONFIGS.items():
    X_set = df_full[FEATURE_SETS[set_name]]
    params = {**hp, **FIXED_PARAMS}

    for seed in SEED_CHECK_SEEDS:
        gss_check = GroupShuffleSplit(n_splits=1, test_size=FINAL_TEST_FRACTION, random_state=seed)
        train_idx, holdout_idx = next(gss_check.split(X_set, y_reg, groups))

        X_train, X_holdout = X_set.iloc[train_idx], X_set.iloc[holdout_idx]
        y_reg_train, y_reg_holdout = y_reg[train_idx], y_reg[holdout_idx]
        y_class_train, y_class_holdout = y_class[train_idx], y_class[holdout_idx]
        g_train = groups[train_idx]

        reg_model_check, _ = fit_with_early_stopping(
            XGBRegressor, X_train, y_reg_train, g_train, params,
            fit_kwargs={"init_kwargs": {"eval_metric": "rmse"}},
        )
        r2_check = r2_score(y_reg_holdout, reg_model_check.predict(X_holdout))

        clf_model_check, _ = fit_with_early_stopping(
            XGBClassifier, X_train, y_class_train, g_train, params,
            fit_kwargs={"init_kwargs": {"eval_metric": "mlogloss", "objective": "multi:softprob"}},
        )
        acc_check = accuracy_score(y_class_holdout, clf_model_check.predict(X_holdout))

        seed_check_rows.append({"feature_set": set_name, "seed": seed,
                                 "r2": r2_check, "accuracy": acc_check})
        print(f"{set_name:16s} | seed={seed}: R2={r2_check:.4f}  Acc={acc_check:.4f}")

seed_check_df = pd.DataFrame(seed_check_rows)
summary = seed_check_df.groupby("feature_set")[["r2", "accuracy"]].agg(["mean", "std"])
print("\n Seed-sensitivity summary (mean +/- std across seeds) ")
print(summary)

out_path_seed = rf"{BASE_PATH}\seed_sensitivity_check.csv"
seed_check_df.to_csv(out_path_seed, index=False)
print(f"\nSaved: {out_path_seed}")

operational_only | seed=101: R2=0.1324  Acc=0.5409
operational_only | seed=102: R2=0.1052  Acc=0.5403
operational_only | seed=103: R2=0.1341  Acc=0.5408
operational_only | seed=104: R2=0.1163  Acc=0.5343
operational_only | seed=105: R2=0.1170  Acc=0.5362
operational_only | seed=106: R2=0.1168  Acc=0.5317
operational_only | seed=107: R2=0.1215  Acc=0.5401
operational_only | seed=108: R2=0.1045  Acc=0.5371
operational_only | seed=109: R2=0.1316  Acc=0.5400
operational_only | seed=110: R2=0.1252  Acc=0.5395
price_and_staff  | seed=101: R2=0.2772  Acc=0.5734
price_and_staff  | seed=102: R2=0.2590  Acc=0.5743
price_and_staff  | seed=103: R2=0.2807  Acc=0.5697
price_and_staff  | seed=104: R2=0.2809  Acc=0.5722
price_and_staff  | seed=105: R2=0.2659  Acc=0.5654
price_and_staff  | seed=106: R2=0.2680  Acc=0.5685
price_and_staff  | seed=107: R2=0.2747  Acc=0.5728
price_and_staff  | seed=108: R2=0.2563  Acc=0.5660
price_and_staff  | seed=109: R2=0.2790  Acc=0.5693
price_and_staff  | seed=110: R2

In [43]:
print("pm_has_prior_wifi_resid x is_channel -- row counts")
print(pd.crosstab(df_full["pm_has_prior_wifi_resid"], df_full["is_channel"],
                   margins=True, margins_name="Total"))

print("\nNPS (mean recommendation score) by is_channel x pm_has_prior_wifi_resid")
print(df_full.groupby(["is_channel", "pm_has_prior_wifi_resid"])[TARGET]
      .agg(["mean", "count"]))

print("\nstaff_composite mean by is_channel x pm_has_prior_wifi_resid")
print(df_full.groupby(["is_channel", "pm_has_prior_wifi_resid"])["staff_composite"]
      .agg(["mean", "count"]))

n_unique = df_full["pm_has_prior_wifi_resid"].nunique()
print(f"\npm_has_prior_wifi_resid unique values: {n_unique}")
if n_unique > 20:
    print("\nLooks continuous, not binary -- describe by is_channel instead:")
    print(df_full.groupby("is_channel")["pm_has_prior_wifi_resid"].describe())

pm_has_prior_wifi_resid x is_channel -- row counts
is_channel                   0       1   Total
pm_has_prior_wifi_resid                       
-0.968896                 1916       0    1916
-0.0                       519  116265  116784
 0.031104                59684       0   59684
 Total                   62119  116265  178384

NPS (mean recommendation score) by is_channel x pm_has_prior_wifi_resid
                                        mean   count
is_channel pm_has_prior_wifi_resid                  
0          -9.688961e-01            7.695198    1916
           -6.006307e-14            7.913295     519
            3.110390e-02            7.755445   59684
1          -6.006307e-14            8.244691  116265

staff_composite mean by is_channel x pm_has_prior_wifi_resid
                                        mean   count
is_channel pm_has_prior_wifi_resid                  
0          -9.688961e-01            4.122129    1916
           -6.006307e-14            4.128773     519
  

In [44]:
import matplotlib.pyplot as plt
import os

FIGURE_DIR = rf"{BASE_PATH}\figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

shap_explainers = {}
shap_samples = {}
shap_values_dict = {}

for set_name in FINAL_CONFIGS:
    X_holdout = final_holdouts[set_name]["X_holdout"]
    reg_model = final_models[(set_name, "reg")]
    explainer = shap.TreeExplainer(reg_model)
    sample_main = X_holdout.sample(n=min(SHAP_SAMPLE_SIZE, len(X_holdout)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample_main)
    shap_explainers[set_name] = explainer
    shap_samples[set_name] = sample_main
    shap_values_dict[set_name] = shap_values
    print(f"{set_name}: SHAP values computed on {len(sample_main)} rows")


def save_dependence_plot(set_name, feature, interaction_feature, filename):
    shap.dependence_plot(
        feature, shap_values_dict[set_name], shap_samples[set_name],
        interaction_index=interaction_feature, show=False
    )
    fig = plt.gcf()
    fig.suptitle(f"{feature} -- {set_name}", fontsize=11, y=1.02)
    out_path = rf"{FIGURE_DIR}\{filename}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")



save_dependence_plot("operational_only", "arrival_delay_minute", None, "dep_arrival_delay_operational")
save_dependence_plot("operational_only", "early_journey_delay_minute", None, "dep_early_journey_operational")


save_dependence_plot("operational_only", "arrival_delay_minute", "had_disruption", "dep_arrival_delay_x_disruption")
save_dependence_plot("price_and_staff", "arrival_delay_minute", "staff_composite", "dep_arrival_delay_x_staff")
save_dependence_plot("price_and_staff", "metadata_price", "staff_composite", "dep_price_x_staff")


for set_name in FINAL_CONFIGS:
    shap.summary_plot(shap_values_dict[set_name], shap_samples[set_name], show=False, max_display=15)
    fig = plt.gcf()
    fig.suptitle(f"SHAP summary -- {set_name}", fontsize=11, y=1.02)
    out_path = rf"{FIGURE_DIR}\shap_summary_{set_name}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")

operational_only: SHAP values computed on 3000 rows
price_and_staff: SHAP values computed on 3000 rows
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_arrival_delay_operational.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_early_journey_operational.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_arrival_delay_x_disruption.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_arrival_delay_x_staff.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_price_x_staff.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\shap_summary_operational_only.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\shap_summary_price_and_staff.png


In [45]:
import matplotlib.pyplot as plt
import pandas as pd
import os

FIGURE_DIR = rf"{BASE_PATH}\figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

clf_int_df = pd.read_csv(rf"{BASE_PATH}\shap_interaction_pairs_classifier.csv")
clf_int_df["pair"] = clf_int_df["feature_1"] + " x " + clf_int_df["feature_2"]

CLASS_ORDER = ["Detractor", "Passive", "Promoter"]
CLASS_COLORS = {"Detractor": "#D85A30", "Passive": "#888780", "Promoter": "#1D9E75"}
TOP_N = 8

for set_name in clf_int_df["feature_set"].unique():
    fig, axes = plt.subplots(3, 1, figsize=(8, 11))
    for ax, cls in zip(axes, CLASS_ORDER):
        sub = (clf_int_df[(clf_int_df["feature_set"] == set_name) & (clf_int_df["class"] == cls)]
               .sort_values("mean_abs_interaction", ascending=True)
               .tail(TOP_N))
        ax.barh(sub["pair"], sub["mean_abs_interaction"], color=CLASS_COLORS[cls])
        ax.set_title(cls, fontsize=11, loc="left")
        ax.tick_params(axis="y", labelsize=8)
    fig.suptitle(f"Top classifier SHAP interaction pairs -- {set_name}", fontsize=12, y=1.0)
    fig.tight_layout()
    out_path = rf"{FIGURE_DIR}\clf_interactions_{set_name}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")

Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\clf_interactions_operational_only.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\clf_interactions_price_and_staff.png


In [46]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

FIGURE_DIR = rf"{BASE_PATH}\figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

seed_df = pd.read_csv(rf"{BASE_PATH}\seed_sensitivity_check.csv")

OFFICIAL = {  
    "operational_only": {"r2": 0.1157, "accuracy": 0.5420},
    "price_and_staff": {"r2": 0.2572, "accuracy": 0.5717},
}

feature_sets = list(FINAL_CONFIGS.keys())
colors = {"operational_only": "#378ADD", "price_and_staff": "#D85A30"}

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
rng = np.random.default_rng(0)

for metric, ax, label in [("r2", axes[0], "Holdout R2"), ("accuracy", axes[1], "Holdout accuracy")]:
    for i, fs in enumerate(feature_sets):
        vals = seed_df.loc[seed_df["feature_set"] == fs, metric]
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, color=colors[fs], alpha=0.7, s=40,
                   label=f"{fs} (seeds 101-110)" if metric == "r2" else None)
        ax.scatter([i], [OFFICIAL[fs][metric]], color=colors[fs], marker="*", s=220,
                   edgecolor="black", linewidth=0.6,
                   label=f"{fs} (official seed 42)" if metric == "r2" else None)
        mean_val = vals.mean()
        ax.hlines(mean_val, i - 0.2, i + 0.2, colors=colors[fs], linestyles="dashed", linewidth=1.2)
    ax.set_xticks(range(len(feature_sets)))
    ax.set_xticklabels(feature_sets, rotation=10)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel(label)

axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Seed sensitivity -- 10-seed sweep vs official seed=42 holdout", fontsize=12, y=1.02)
fig.tight_layout()
out_path = rf"{FIGURE_DIR}\seed_sensitivity.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")

Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\seed_sensitivity.png


In [47]:
import matplotlib.pyplot as plt
import numpy as np
import os

FIGURE_DIR = rf"{BASE_PATH}\figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

grp = (df_full.groupby(["is_channel", "pm_has_prior_wifi_resid"])[TARGET]
       .agg(["mean", "count"]).reset_index())
grp = grp[grp["count"] > 0]


def label_group(row):
    if row["is_channel"] == 1:
        return f"Channel\n(n={int(row['count']):,})"
    resid = row["pm_has_prior_wifi_resid"]
    if np.isclose(resid, -0.968896, atol=1e-3):
        return f"Continental,\nno prior wifi\n(n={int(row['count']):,})"
    elif np.isclose(resid, 0.031104, atol=1e-3):
        return f"Continental,\nhad prior wifi\n(n={int(row['count']):,})"
    else:
        return f"Continental,\nmissing/placeholder\n(n={int(row['count']):,})"


grp["group_label"] = grp.apply(label_group, axis=1)
grp = grp.sort_values("is_channel")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#378ADD" if c == 1 else "#D85A30" for c in grp["is_channel"]]
ax.bar(grp["group_label"], grp["mean"], color=colors)
ax.set_ylabel("Mean recommendation score (0-10)")
ax.set_title("NPS by channel and prior-wifi history", fontsize=11)
ax.set_ylim(0, 10)
for i, m in enumerate(grp["mean"]):
    ax.text(i, m + 0.15, f"{m:.2f}", ha="center", fontsize=10)
fig.tight_layout()
out_path = rf"{FIGURE_DIR}\nps_by_channel_wifi_history.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")

Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\nps_by_channel_wifi_history.png


In [48]:
import matplotlib.pyplot as plt
import os

FIGURE_DIR = rf"{BASE_PATH}\figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

if "shap_values_dict" not in globals():
    shap_explainers, shap_samples, shap_values_dict = {}, {}, {}
    for set_name in FINAL_CONFIGS:
        X_holdout = final_holdouts[set_name]["X_holdout"]
        reg_model = final_models[(set_name, "reg")]
        explainer = shap.TreeExplainer(reg_model)
        sample_main = X_holdout.sample(n=min(SHAP_SAMPLE_SIZE, len(X_holdout)), random_state=RANDOM_STATE)
        shap_explainers[set_name] = explainer
        shap_samples[set_name] = sample_main
        shap_values_dict[set_name] = explainer.shap_values(sample_main)
        print(f"{set_name}: SHAP values rebuilt on {len(sample_main)} rows")


def save_dependence_plot(set_name, feature, interaction_feature, filename, mask=None, title_suffix=""):
    sv = shap_values_dict[set_name]
    sample = shap_samples[set_name]
    if mask is not None:
        sv = sv[mask.values]
        sample = sample[mask]
    shap.dependence_plot(feature, sv, sample, interaction_index=interaction_feature, show=False)
    fig = plt.gcf()
    fig.suptitle(f"{feature} -- {set_name}{title_suffix}", fontsize=11, y=1.02)
    out_path = rf"{FIGURE_DIR}\{filename}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")



save_dependence_plot("price_and_staff", "staff_composite", None, "dep_staff_composite")

channel_mask = shap_samples["operational_only"]["is_channel"] == 1
save_dependence_plot("operational_only", "clean_score_routine", None,
                      "dep_clean_score_channel_only", mask=channel_mask, title_suffix=" (Channel only)")

save_dependence_plot("operational_only", "clean_score_routine", None,
                      "dep_clean_score_pooled_DIAGNOSTIC_ONLY", title_suffix=" (pooled, do not present)")


save_dependence_plot("operational_only", "total_open_faults", None, "dep_total_open_faults")
save_dependence_plot("operational_only", "pm_days_since_catering", None, "dep_pm_days_catering")
save_dependence_plot("operational_only", "pm_days_since_interior", None, "dep_pm_days_interior")

save_dependence_plot("operational_only", "arrival_delay_minute", "delay_is_internal",
                      "dep_arrival_delay_x_internal_CHECK")

Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_staff_composite.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_clean_score_channel_only.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_clean_score_pooled_DIAGNOSTIC_ONLY.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_total_open_faults.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_pm_days_catering.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_pm_days_interior.png
Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\dep_arrival_delay_x_internal_CHECK.png


In [49]:
fleet_data = {
    "TGH (older fleet)": {"delay": 16.66, "nps": 7.727},
    "RUB (newer fleet)": {"delay": 19.03, "nps": 8.018},
}
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
fleets = list(fleet_data.keys())
colors = ["#D85A30", "#1D9E75"]

axes[0].bar(fleets, [fleet_data[f]["delay"] for f in fleets], color=colors)
axes[0].set_title("Mean arrival delay (min)", fontsize=11)
for i, f in enumerate(fleets):
    axes[0].text(i, fleet_data[f]["delay"] + 0.3, f"{fleet_data[f]['delay']:.2f}", ha="center")

axes[1].bar(fleets, [fleet_data[f]["nps"] for f in fleets], color=colors)
axes[1].set_title("Mean recommendation score", fontsize=11)
axes[1].set_ylim(0, 10)
for i, f in enumerate(fleets):
    axes[1].text(i, fleet_data[f]["nps"] + 0.15, f"{fleet_data[f]['nps']:.2f}", ha="center")

fig.suptitle("Fleet age effect: TGH penalised despite less delay\n(fleet age per Neil's domain input, not derived from data)",
             fontsize=10, y=1.05)
fig.tight_layout()
out_path = rf"{FIGURE_DIR}\fleet_comparison_tgh_rub.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")

Saved: C:\Users\gehan\Downloads\thesis\extracted tables\last update data\figures\fleet_comparison_tgh_rub.png


In [52]:
print(shap_samples["operational_only"].loc[channel_mask, "clean_score_routine"].value_counts().head())

clean_score_routine
100.0000    89
0.0000      27
99.1655     20
84.7115     15
91.9425     14
Name: count, dtype: int64


In [53]:
print(df_full.loc[df_full["cause_No_Delay"] == 1, "arrival_delay_minute"].describe())

count    68932.000000
mean         3.822622
std         17.092617
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        319.000000
Name: arrival_delay_minute, dtype: float64


In [54]:
def band_faults(x):
    if x == 0: return "0"
    if x < 35: return "1-34"
    if x < 58: return "35-57 (the dip)"
    if x < 65: return "58-64 (the spike)"
    return "65+"

df_full["fault_band"] = df_full["total_open_faults"].apply(band_faults)
band_order = ["0", "1-34", "35-57 (the dip)", "58-64 (the spike)", "65+"]

for fleet_col in ["equip_TGH", "equip_RUB", "equip_E320", "is_channel"]:
    print(f"\n=== total_open_faults band x {fleet_col} (row %) ===")
    print((pd.crosstab(df_full["fault_band"], df_full[fleet_col], normalize="index") * 100)
          .round(1).reindex(band_order))

print("\n=== mean recommendation score by fault_band ===")
print(df_full.groupby("fault_band")[TARGET].agg(["mean", "count"]).reindex(band_order))


def band_catering(x):
    if pd.isna(x): return "missing"
    if x < 25: return "0-24"
    if x < 65: return "25-64 (the dip)"
    if x < 85: return "65-84"
    return "85+"

df_full["catering_band"] = df_full["pm_days_since_catering"].apply(band_catering)
band_order_cat = ["0-24", "25-64 (the dip)", "65-84", "85+", "missing"]

for fleet_col in ["equip_TGH", "equip_RUB", "equip_E320", "is_channel"]:
    print(f"\n=== pm_days_since_catering band x {fleet_col} (row %) ===")
    print((pd.crosstab(df_full["catering_band"], df_full[fleet_col], normalize="index") * 100)
          .round(1).reindex(band_order_cat))

print("\n mean recommendation score by catering_band ")
print(df_full.groupby("catering_band")[TARGET].agg(["mean", "count"]).reindex(band_order_cat))


=== total_open_faults band x equip_TGH (row %) ===
equip_TGH          False  True 
fault_band                     
0                  100.0    0.0
1-34                12.2   87.8
35-57 (the dip)      9.5   90.5
58-64 (the spike)   10.6   89.4
65+                 10.5   89.5

=== total_open_faults band x equip_RUB (row %) ===
equip_RUB          False  True 
fault_band                     
0                  100.0    0.0
1-34                87.8   12.2
35-57 (the dip)     90.5    9.5
58-64 (the spike)   89.4   10.6
65+                 89.5   10.5

=== total_open_faults band x equip_E320 (row %) ===
equip_E320         False  True 
fault_band                     
0                   21.6   78.4
1-34               100.0    0.0
35-57 (the dip)    100.0    0.0
58-64 (the spike)  100.0    0.0
65+                100.0    0.0

=== total_open_faults band x is_channel (row %) ===
is_channel             0     1
fault_band                    
0                    1.6  98.4
1-34               100.0 